# Goal

Create a clean, unified dataset of e-commerce reviews from multiple platforms, ready for NLP tasks.

# 1. Data Exploration 

# 

In [1]:
import os

def explore(path, indent=0):
    try:
        items = os.listdir(path)
        for item in items:
            full = os.path.join(path, item)
            if os.path.isdir(full):
                print('  ' * indent + f'{item}/')
                explore(full, indent + 1)
            else:
                mb = os.path.getsize(full) / 1e6
                print('  ' * indent + f'└── {item}  ({mb:.1f} MB)')
    except:
        print('  ' * indent + ' cannot read')

explore('/kaggle/input')

datasets/
  bennjimatakwa/
    ecommerce-customerbehavior-dl/
      csv/
        └── Alibaba.csv  (12.8 MB)
        └── Aliexpress.csv  (25.7 MB)
        └── Myntra.csv  (16.6 MB)
        └── Flipkart.csv  (7.6 MB)
        └── Meesho.csv  (22.6 MB)
        └── Lazada.csv  (17.7 MB)
        └── Amazon shopping.csv  (23.8 MB)
        └── Snapdeal.csv  (11.1 MB)
        └── Shein.csv  (14.1 MB)
        └── Daraz Online Shopping App.csv  (15.1 MB)
        └── Walmart.csv  (17.6 MB)


In [2]:
import pandas as pd
import os
from pathlib import Path
RAW_DIR  = Path('/kaggle/input/datasets/bennjimatakwa/ecommerce-customerbehavior-dl/csv')
PROC_DIR = Path('/kaggle/working/data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROC_DIR / 'merged_reviews.csv'

print('Paths set')
print(f'   Input  : {RAW_DIR}')
print(f'   Output : {OUTPUT_FILE}')

Paths set
   Input  : /kaggle/input/datasets/bennjimatakwa/ecommerce-customerbehavior-dl/csv
   Output : /kaggle/working/data/processed/merged_reviews.csv


# 2. Data Standardization

In [3]:
EXPECTED_COLUMNS = [
    'reviewId', 'content', 'score',
    'thumbsUpCount', 'at',
    'replyContent', 'repliedAt',
    'appName'
]

csv_files = list(RAW_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files:\n')

df_list = []

for file in csv_files:
    df = pd.read_csv(file)
    print(f'   {file.name:<45} {df.shape[0]:>8,} rows  {df.shape[1]} cols')

    # Add missing columns
    for col in EXPECTED_COLUMNS:
        if col not in df.columns:
            df[col] = None

    # Keep only expected columns
    df = df[EXPECTED_COLUMNS]

    # Fill appName if missing
    if df['appName'].isnull().all():
        df['appName'] = file.stem

    df_list.append(df)

# ── Merge
merged_df = pd.concat(df_list, ignore_index=True)

print(f'\n{"─"*55}')
print(f'  Total rows    : {len(merged_df):,}')
print(f'  Total columns : {merged_df.shape[1]}')
print(f'  Platforms     : {merged_df["appName"].nunique()}')
print(f'  Platform list : {sorted(merged_df["appName"].unique())}')

Found 11 CSV files:

   Alibaba.csv                                     94,500 rows  8 cols
   Aliexpress.csv                                 126,000 rows  8 cols
   Myntra.csv                                      36,000 rows  8 cols
   Flipkart.csv                                    18,000 rows  8 cols
   Meesho.csv                                      36,000 rows  8 cols
   Lazada.csv                                      54,000 rows  8 cols
   Amazon shopping.csv                             99,000 rows  8 cols
   Snapdeal.csv                                    27,000 rows  8 cols
   Shein.csv                                       40,500 rows  8 cols
   Daraz Online Shopping App.csv                   54,000 rows  8 cols
   Walmart.csv                                     45,000 rows  8 cols

───────────────────────────────────────────────────────
  Total rows    : 630,000
  Total columns : 8
  Platforms     : 11
  Platform list : ['Alibaba', 'Aliexpress', 'Amazon shopping', 'Daraz Onli

# 3. Data Validation & Quality Checks

In [4]:
print('Validation checks:')
print(f'  Shape          : {merged_df.shape}')
print(f'  Null content   : {merged_df["content"].isnull().sum():,}')
print(f'  Null score     : {merged_df["score"].isnull().sum():,}')
print(f'  Score range    : {merged_df["score"].min()} → {merged_df["score"].max()}')
print(f'  Duplicates     : {merged_df.duplicated().sum():,}')

print('\n  Reviews per platform:')
counts = merged_df['appName'].value_counts()
for platform, count in counts.items():
    pct = count / len(merged_df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {platform:<45} {count:>8,}  ({pct:4.1f}%)  {bar}')

print(f'\n  dtypes:')
print(merged_df.dtypes)

Validation checks:
  Shape          : (630000, 8)
  Null content   : 1
  Null score     : 0
  Score range    : 1 → 5
  Duplicates     : 0

  Reviews per platform:
  Aliexpress                                     126,000  (20.0%)  ██████████
  Amazon shopping                                 99,000  (15.7%)  ███████
  Alibaba                                         94,500  (15.0%)  ███████
  Daraz Online Shopping App                       54,000  ( 8.6%)  ████
  Lazada                                          54,000  ( 8.6%)  ████
  Walmart                                         45,000  ( 7.1%)  ███
  Shein                                           40,500  ( 6.4%)  ███
  Myntra                                          36,000  ( 5.7%)  ██
  Meesho                                          36,000  ( 5.7%)  ██
  Snapdeal                                        27,000  ( 4.3%)  ██
  Flipkart                                        18,000  ( 2.9%)  █

  dtypes:
reviewId          object
content 

# 4.Remove duplicates

In [5]:
# ── Remove duplicates before saving
before = len(merged_df)
merged_df = merged_df.drop_duplicates(subset=['reviewId'], keep='first')
merged_df = merged_df.reset_index(drop=True)
after = len(merged_df)

print(f'  Duplicates removed : {before - after:,}')
print(f'  Final rows         : {after:,}')

# ── Save
merged_df.to_csv(OUTPUT_FILE, index=False)
size_mb = OUTPUT_FILE.stat().st_size / 1e6
print(f'\n Saved → {OUTPUT_FILE}')
print(f'   Size : {size_mb:.1f} MB')
print(f'   Rows : {after:,}')
print(f'   Cols : {merged_df.shape[1]}')

  Duplicates removed : 0
  Final rows         : 630,000

 Saved → /kaggle/working/data/processed/merged_reviews.csv
   Size : 184.2 MB
   Rows : 630,000
   Cols : 8


In [6]:
print('Sample rows:')
print(merged_df[['appName', 'score', 'content', 'thumbsUpCount', 'at']].sample(5, random_state=42).to_string())
print(f'   merged_reviews.csv saved to /kaggle/working/data/processed/')

Sample rows:
           appName  score                                                                                                                                                                                                                                                                                                                                                                                        content  thumbsUpCount             at
364426      Lazada      1                                                                                                                                                                                                                                                                                                                 Having problem to see the option we choose after payment is paid. Please help.              6  1578574364000
224752      Myntra      1                                                                                    